In [0]:
dbutils.widgets.help()

In [0]:
dbutils.widgets.text("p_data_source","")
v_data_source = dbutils.widgets.get("p_data_source")


In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
#circuits_df=spark.read.csv("dbfs:/mnt/formula1dl4dbcourse/raw/circuits.csv")
#circuits_df=spark.read.option("header",True).option("inferSchema", True).csv("dbfs:/mnt/formula1dl4dbcourse/raw/circuits.csv")
circuits_df=spark.read.option("header",True).option("inferSchema", True).csv(f"{raw_folder_path}/circuits.csv")



In [0]:
type(circuits_df)

In [0]:
circuits_df.show()


In [0]:
display(dbutils.fs.mounts())

In [0]:
def add_value(a,b):
    return a+b

add_value(1,2)


In [0]:
#circuits_df.printSchema()

circuits_df.describe().show()


In [0]:
display(circuits_df)

### select only the required columns

In [0]:
circuits__selected_df = circuits_df.select("circuitId","circuitRef","name","location","country","lat","lng","alt")
display(circuits__selected_df)

#circuits__selected_df = circuits_df.select(circuits_df.circuitId,circuits_df.circuitRef,circuits_df.name,circuits_df.location,circuits_df.country,

In [0]:
from pyspark.sql.functions import col
circuits__selected_df = circuits_df.select(col("circuitId"),col("circuitRef"),col("name"),col("location"),col("country"),col("lat"),col("lng"),col("alt"))
display(circuits__selected_df)


In [0]:
from pyspark.sql.functions import current_timestamp, lit
circuits_renamed_df = circuits__selected_df.withColumnRenamed("circuitId","circuit_id") \
.withColumnRenamed("circuitRef","circuit_ref") \
.withColumnRenamed("lat","latitude") \
.withColumnRenamed("lng","longitude") \
.withColumnRenamed("alt","altitude") \
.withColumn("data_source",lit(v_data_source))

display(circuits_renamed_df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
circuits_final_df = add_ingestion_date(circuits_renamed_df)

display(circuits_final_df)
#circuits_final_df.write.mode("overwrite")

### write the data to datalake as parquet

In [0]:
#circuits_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/circuits")
circuits_final_df.write.mode("overwrite").format("parquet").saveAsTable("f1_processed.circuits")

In [0]:
%fs
ls /mnt/formula1dl4dbcourse/procesed/circuits

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/circuits")
display(df)

In [0]:
dbutils.notebook.exit("Success")

In [0]:
%sql
select * from f1_processed.circuits